# English LLM Benchmark Suite

> **Based on:** [hyogrin/rhoai-lmeval-builder-lab](https://github.com/hyogrin/rhoai-lmeval-builder-lab) (EvalHub SDK pattern)  
> **Benchmarks:** ARC Easy, TruthfulQA, HellaSwag, GSM8K, MMLU  
> **Adapted by:** [gymnatics/RHOAI-Toolkit](https://github.com/gymnatics/RHOAI-Toolkit)

### Provider Selection

EvalHub supports multiple evaluation providers. This notebook uses **`lm_evaluation_harness`** which:
- Creates **MLflow runs** with full metric tracking
- Supports **all benchmark types** (generative AND multiple-choice)
- Requires an **external model route** (standard TLS, not internal service-ca)
- Requires a **tokenizer** parameter pointing to the HuggingFace model ID

| Benchmark | Type | Dataset Size | Primary Metric |
|-----------|------|-------------|----------------|
| **arc_easy** | Multiple-choice | 2,376 | acc_norm |
| **truthfulqa_mc1** | Multiple-choice | 817 | mc1_acc |
| **hellaswag** | Multiple-choice | 10,042 | acc_norm |
| **gsm8k** | Generative (math) | 1,319 | exact_match |
| **mmlu** | Multiple-choice | 15,908 | acc_norm |

**Note:** The `lighteval` provider is faster for generative benchmarks but does NOT create MLflow runs
(adapter v0.2.0 limitation) and has metric name mismatches with EvalHub's registry.

In [ ]:
!pip install -q "eval-hub-sdk[client]"

In [ ]:
# Auto-configure environment for this cluster
# For lm_evaluation_harness: we need the EXTERNAL route URL (proper TLS) and a tokenizer
import os
import subprocess

try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".env"), override=True)
except ImportError:
    pass

NAMESPACE = os.environ.get("NAMESPACE", "lmeval-demo")
EVALHUB_URL = os.environ.get("EVALHUB_URL", "https://evalhub.redhat-ods-applications.svc:8443")

# For lm_evaluation_harness we need:
# 1. External route URL (not internal service-ca HTTPS) — DIRECT_ROUTE_URL
# 2. model.name = vLLM served_model_name (used in API requests) — DIRECT_MODEL_NAME
# 3. tokenizer = HuggingFace model ID (for tokenizer download)
MODEL_ROUTE_URL = os.environ.get("DIRECT_ROUTE_URL", "")
MODEL_SERVED_NAME = os.environ.get("DIRECT_MODEL_NAME", "")
TOKENIZER_HF_ID = os.environ.get("TOKENIZER_HF_ID", "Qwen/Qwen3-8B")

# Fallback: auto-detect from InferenceService status.url
if not MODEL_ROUTE_URL or not MODEL_SERVED_NAME:
    try:
        result = subprocess.run(
            ["oc", "get", "inferenceservice", "-A", "--no-headers",
             "-o", "custom-columns=NS:.metadata.namespace,NAME:.metadata.name,URL:.status.url"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.strip().split("\n"):
                parts = line.split()
                if len(parts) >= 3 and "qwen" in parts[1].lower():
                    MODEL_SERVED_NAME = MODEL_SERVED_NAME or parts[1]
                    route_url = parts[2].rstrip("/")
                    MODEL_ROUTE_URL = MODEL_ROUTE_URL or f"{route_url}/v1"
                    break
    except Exception:
        pass

print(f"Namespace:       {NAMESPACE}")
print(f"Model Name:      {MODEL_SERVED_NAME}")
print(f"Model Route URL: {MODEL_ROUTE_URL}")
print(f"Tokenizer (HF):  {TOKENIZER_HF_ID}")
print(f"EvalHub:         {EVALHUB_URL}")

if not MODEL_ROUTE_URL:
    print("\n⚠ No external model route detected.")
    print("  Deploy a model as InferenceService (not LLMInferenceService) for external route access.")
    print("  Or set DIRECT_ROUTE_URL and DIRECT_MODEL_NAME env vars.")

## Step 1: Configuration

Using `lm_evaluation_harness` provider with the external model route for proper MLflow tracking.

In [ ]:
import subprocess

# Auth token: prefer injected env var (evalhub-service SA token), fall back to oc whoami -t
EVALHUB_AUTH_TOKEN = os.environ.get("EVALHUB_AUTH_TOKEN", "")
if not EVALHUB_AUTH_TOKEN:
    _r = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    EVALHUB_AUTH_TOKEN = _r.stdout.strip() if _r.returncode == 0 else None

# Provider: lm_evaluation_harness for full MLflow support
PROVIDER_ID = "lm_evaluation_harness"

print(f"Namespace:       {NAMESPACE}")
print(f"Model Name:      {MODEL_SERVED_NAME}")
print(f"Model Route:     {MODEL_ROUTE_URL}")
print(f"Tokenizer:       {TOKENIZER_HF_ID}")
print(f"Provider:        {PROVIDER_ID}")
print(f"EvalHub URL:     {EVALHUB_URL}")
print(f"Auth token:      {'✓ set' if EVALHUB_AUTH_TOKEN else '✗ missing'}")

## Step 2: Initialize the EvalHub Client

In [ ]:
from evalhub import (
    SyncEvalHubClient,
    ModelConfig,
    BenchmarkConfig,
    JobSubmissionRequest,
    ExperimentConfig,
    ExperimentTag,
    JobStatus,
)

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=EVALHUB_AUTH_TOKEN,
    insecure=True,
    tenant=NAMESPACE,
)

model = ModelConfig(
    url=MODEL_ROUTE_URL,
    name=MODEL_SERVED_NAME,
)

print(f"EvalHub client connected: {EVALHUB_URL}")
print(f"Model target:             {model.name} @ {model.url}")

## Step 3: Discover Available Benchmarks

Query EvalHub for benchmarks available through the `lm_evaluation_harness` provider.
All benchmark types (generative and multiple-choice) are supported with proper MLflow tracking.

In [ ]:
# Benchmarks to run (registered in lm_evaluation_harness provider)
TARGET_BENCHMARKS = [
    "arc_easy",        # Basic science reasoning (2,376 questions)
    "truthfulqa_mc1",  # Truthfulness multiple-choice (817 questions)
]

# Patch SDK model: EvalHub server may omit 'description'/'category' fields
# but eval-hub-sdk 0.1.6 Pydantic model marks them as required.
# Must rebuild entire model chain: Benchmark -> Provider -> ProviderList
from evalhub.models.api import Benchmark as _Benchmark, Provider as _Provider, ProviderList as _ProviderList
for _field_name in ("description", "category"):
    if _field_name in _Benchmark.model_fields:
        _Benchmark.model_fields[_field_name].default = None
        _Benchmark.model_fields[_field_name].annotation = str | None
_Benchmark.model_rebuild(force=True)
_Provider.model_rebuild(force=True)
_ProviderList.model_rebuild(force=True)

all_benchmarks = client.benchmarks.list(provider_id=PROVIDER_ID)

print(f"Total {PROVIDER_ID} benchmarks: {len(all_benchmarks)}")
print(f"\n{'='*70}")
print("TARGET BENCHMARKS FOR THIS EVALUATION:")
print(f"{'='*70}")

target_benchmarks = [bm for bm in all_benchmarks if bm.id in TARGET_BENCHMARKS]
other_benchmarks = [bm for bm in all_benchmarks if bm.id not in TARGET_BENCHMARKS]

for bm in target_benchmarks:
    metrics_str = ", ".join(bm.metrics[:4]) if bm.metrics else "N/A"
    print(f"  ✓ {bm.id:25s} metrics=[{metrics_str}]")

if other_benchmarks:
    print(f"\n{'='*70}")
    print("OTHER AVAILABLE BENCHMARKS:")
    print(f"{'='*70}")
    for bm in other_benchmarks:
        metrics_str = ", ".join(bm.metrics[:3]) if bm.metrics else "N/A"
        print(f"  {bm.id:25s} metrics=[{metrics_str}]")

---

## Evaluation 1: Quick Benchmark (AIME 2024)

Start with AIME 2024 (30 competition math problems) to verify the pipeline works end-to-end.
This is the fastest available generative benchmark (~10-15 min depending on model output length).

### 1-A: Submit the Job

In [ ]:
single_request = JobSubmissionRequest(
    name="arc-easy-eval",
    description=f"ARC Easy science reasoning (limit=20) - {MODEL_SERVED_NAME}",
    tags=["english", "arc", "reasoning", MODEL_SERVED_NAME],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="arc_easy",
            provider_id=PROVIDER_ID,
            parameters={"num_fewshot": 0, "limit": 20, "tokenizer": TOKENIZER_HF_ID},
        ),
    ],
    experiment=ExperimentConfig(
        name="english-arc-easy-eval",
        tags=[
            ExperimentTag(key="language", value="english"),
            ExperimentTag(key="benchmark_suite", value="arc_easy"),
            ExperimentTag(key="model", value=MODEL_SERVED_NAME),
        ],
    ),
)

job = client.jobs.submit(single_request)

print(f"Job submitted successfully!")
print(f"  Job ID:        {job.id}")
print(f"  Name:          {job.name}")
print(f"  State:         {job.state.value}")
print(f"  MLflow Exp ID: {job.resource.mlflow_experiment_id or 'pending'}")
print(f"\n  Expected runtime: ~5-6 min (limit=20, multiple-choice)")
print(f"  MLflow run will be created on completion ✓")

### 1-B: Monitor Progress

In [ ]:
import time
from datetime import datetime

TERMINAL_STATES = {
    JobStatus.COMPLETED,
    JobStatus.FAILED,
    JobStatus.CANCELLED,
}


def wait_for_job(client, job_id, poll_interval=15, max_wait=900):
    """Poll job status until terminal state or timeout."""
    start = time.time()
    print(f"Monitoring job {job_id}...")
    print("-" * 70)

    state = JobStatus.PENDING
    elapsed = 0
    last_state = None
    while time.time() - start < max_wait:
        status = client.jobs.get(job_id)
        state = status.state
        elapsed = int(time.time() - start)

        msg = ""
        if status.status and status.status.message:
            msg = f" | {status.status.message.message}"

        bm_info = ""
        if status.status and status.status.benchmarks:
            bm_states = [f"{b.id}={b.status.value}" for b in status.status.benchmarks]
            bm_info = f" | benchmarks: {', '.join(bm_states)}"

        # Print on state change or every 60s
        if state != last_state or elapsed % 60 < poll_interval:
            print(f"  [{elapsed:>4d}s] {state.value:>16s}{msg}{bm_info}")
            last_state = state

        if state in TERMINAL_STATES:
            break

        time.sleep(poll_interval)

    print("-" * 70)
    if state in TERMINAL_STATES:
        print(f"Final state: {state.value} (elapsed: {elapsed}s)")
    else:
        print(f"Timed out after {elapsed}s (state: {state.value})")
        print(f"Job is still running. Check back with: client.jobs.get('{job_id}')")
    return status


completed_job = wait_for_job(client, job.id)

### 1-C: View Results

In [ ]:
def display_job_results(job):
    """Display evaluation results with MLflow links."""
    if not job.results:
        print("No results available.")
        return

    print("Evaluation Results")
    print("=" * 70)

    if job.results.mlflow_experiment_url:
        print(f"\n  MLflow Experiment: {job.results.mlflow_experiment_url}")

    for bm in job.results.benchmarks:
        print(f"\n  Benchmark: {bm.id}")
        print(f"  Provider:  {bm.provider_id}")
        if bm.mlflow_run_id:
            print(f"  MLflow Run: {bm.mlflow_run_id}")

        if bm.metrics:
            print(f"  Metrics:")
            for name, value in bm.metrics.items():
                if isinstance(value, float):
                    print(f"    {name:30s} = {value:.4f}")
                else:
                    print(f"    {name:30s} = {value}")
        else:
            print("  Metrics: (none)")


display_job_results(completed_job)

---

## Evaluation 2: GSM8K Math Benchmark (Optional)

Run the GSM8K grade-school math benchmark (1,319 word problems). This is a standard Open LLM Leaderboard
benchmark using generative evaluation. Expected runtime: ~30-60 min.

In [ ]:
suite_request = JobSubmissionRequest(
    name=f"truthfulqa-eval-{MODEL_SERVED_NAME}",
    description=f"TruthfulQA MC1 for {MODEL_SERVED_NAME} (817 questions)",
    tags=["english", "truthfulqa", "truthfulness", MODEL_SERVED_NAME],
    model=model,
    benchmarks=[
        BenchmarkConfig(
            id="truthfulqa_mc1",
            provider_id=PROVIDER_ID,
            parameters={"num_fewshot": 0, "limit": 50, "tokenizer": TOKENIZER_HF_ID},
        ),
    ],
    experiment=ExperimentConfig(
        name=f"english-truthfulqa-{MODEL_SERVED_NAME}",
        tags=[
            ExperimentTag(key="language", value="english"),
            ExperimentTag(key="benchmark_suite", value="truthfulqa_mc1"),
            ExperimentTag(key="model", value=MODEL_SERVED_NAME),
        ],
    ),
)

suite_job = client.jobs.submit(suite_request)

print(f"TruthfulQA job submitted!")
print(f"  Job ID:     {suite_job.id}")
print(f"  Name:       {suite_job.name}")
print(f"  State:      {suite_job.state.value}")
print(f"\n  Expected runtime: ~5-10 min (limit=50, multiple-choice)")
print(f"  MLflow run will be created on completion ✓")
print(f"\nMonitoring (will time out after 30 min)...")

suite_result = wait_for_job(client, suite_job.id, poll_interval=30, max_wait=1800)
display_job_results(suite_result)

---

## Step 4: Job Management

List, inspect, and manage all evaluation jobs.

### List All Jobs

In [ ]:
jobs_list = client.jobs.list()

print(f"Total Jobs: {len(jobs_list)}")
print("=" * 90)
print(f"{'State':>16s}  {'Job ID':12s}  {'Name':30s}  {'Experiment':25s}  Benchmarks")
print("-" * 90)
for j in jobs_list:
    state = j.state.value
    exp = j.experiment.name if j.experiment else "N/A"
    bms = [b.id for b in j.benchmarks] if j.benchmarks else []
    print(f"{state:>16s}  {j.id[:12]:12s}  {j.name[:30]:30s}  {exp[:25]:25s}  {bms}")

### Inspect a Specific Job

Replace `JOB_ID` with the job you want to inspect.

In [ ]:
# Replace with actual job ID:
# JOB_ID = "your-job-id-here"
# inspected = client.jobs.get(JOB_ID)
# display_job_results(inspected)

---

## Step 5: Export Results

### Export to Markdown

In [ ]:
def collect_results_table(client):
    """Collect benchmark metrics from completed jobs into a comparison dict."""
    jobs_list = client.jobs.list()
    table = {}
    for j in jobs_list:
        if j.state != JobStatus.COMPLETED or not j.results:
            continue
        for bm in j.results.benchmarks:
            if bm.id not in table:
                table[bm.id] = {}
            table[bm.id][j.name] = bm.metrics
    return table


comparison = collect_results_table(client)


def results_to_markdown(comparison, title="EvalHub Benchmark Results"):
    """Convert comparison results to markdown table."""
    if not comparison:
        return "No results available."

    lines = [f"## {title}", ""]

    for benchmark_id, job_results in comparison.items():
        lines.append(f"### {benchmark_id}")
        lines.append("")

        all_metrics = set()
        for metrics in job_results.values():
            all_metrics.update(metrics.keys())
        all_metrics = sorted(all_metrics)

        job_names = sorted(job_results.keys())
        header = "| Metric | " + " | ".join(job_names) + " |"
        sep = "|---" + "|---" * len(job_names) + "|"
        lines.extend([header, sep])

        for metric in all_metrics:
            row = f"| {metric} |"
            for job_name in job_names:
                val = job_results.get(job_name, {}).get(metric, "-")
                if isinstance(val, float):
                    row += f" {val:.4f} |"
                else:
                    row += f" {val} |"
            lines.append(row)

        lines.append("")

    return "\n".join(lines)


md = results_to_markdown(comparison)
print(md)

### Save Results to JSON

In [ ]:
import json
from pathlib import Path

results_root = Path("../results")

jobs_list = client.jobs.list()
saved = 0
for j in jobs_list:
    if j.state != JobStatus.COMPLETED or not j.results:
        continue

    model_name = j.model.name if j.model else "unknown"
    model_dir = results_root / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    job_data = {
        "job_id": j.id,
        "name": j.name,
        "model": {"url": j.model.url, "name": j.model.name},
        "experiment": j.experiment.name if j.experiment else None,
        "benchmarks": [
            {
                "id": bm.id,
                "provider_id": bm.provider_id,
                "metrics": bm.metrics,
                "mlflow_run_id": bm.mlflow_run_id,
            }
            for bm in j.results.benchmarks
        ],
    }

    filename = f"{j.name}_{j.id[:8]}.json"
    output_path = model_dir / filename
    with open(output_path, "w") as f:
        json.dump(job_data, f, indent=2, default=str)
    print(f"Saved: {output_path}")
    saved += 1

print(f"\n{saved} result(s) saved to {results_root.resolve()}/<model-name>/")

---

## Summary

This notebook demonstrated English LLM benchmarking via EvalHub:

1. **Configuration** and EvalHub client setup with auto-detected model
2. **Quick benchmark** (AIME 2024, 30 problems) with MLflow tracking
3. **Full benchmark** (GSM8K, 1,319 math problems)
4. **Job management** -- list and inspect jobs
5. **Export** -- Markdown and JSON output

### Benchmark Compatibility (LightEval via SDK)

The EvalHub SDK uses LightEval with a LiteLLM endpoint adapter. This adapter only
supports **generative** benchmarks. MC benchmarks requiring log-probabilities will fail
with `NotImplementedError: loglikelihood`.

| Works via SDK | Use Dashboard UI Instead |
|---------------|--------------------------|
| gsm8k (math generation) | MMLU, ARC, HellaSwag |
| aime24/aime25 (competition math) | Winogrande, TruthfulQA MC |

For MC benchmarks, use the **EvalHub Dashboard UI** (Develop and train > Evaluations)
which uses the `lm_evaluation_harness` provider with proper tokenizer and logprobs support.

### Next Steps

- **guidellm-benchmark.ipynb** -- GuideLLM throughput/latency profiling
- **korean-mcq-benchmark.ipynb** -- Korean language evaluation
